# Functions and libraries

In previous work with Python, we have been mainly cutting and pasting code whenever we wanted to repeat a particular calculation or perhaps putting it within a loop if we wanted to perform the calculation over a systematically changing set of conditions.

In this notebook, we introduce the use of functions.  Functions are explained in more detail in the Jupyter notebook [Functions in Python](https://colab.research.google.com/github/mjksill/CP213-online/blob/master/notebooks/python_functions.ipynb), which can be found in the learning materials for Week 2 of the Myplace site.

To illustrate the use of functions, we examine the problem of determining the heat of combustion of hydrogen:
$$
{\rm H_2} + \frac{1}{2} {\rm O}_2
⟶
{\rm H_2O}
$$
We want to know the heat $Q$ released when a stream of hydrogen and oxygen initially at $300\,{\rm K}$ is fed into a reactor with an output stream of water vapor at some temperature $T$.

To perform the calculations, the heat capacity $C_p$ is approximated as
$$
C_p(T) = a + b T
$$
where $T$ is the absolute temperature, and $a$ & $b$ are constant parameters.  The corresponding expression for the enthalpy is
$$
H(T) = H_f + a(T-T_0)
+ \frac{b}{2} (T^2-T_0^2)
$$
where $H_f$ is the enthalpy of formation, and $T_0=298.15\,{\rm K}$ is the reference temperature.

Below, the data are entered into Python as a dictionary.

In [ ]:
data = {}
data['H2O'] = {'Hf': -242.20, 'a': 32.47, 'b': 0.0086}     # information for H2O
data['H2']  =	{'Hf':    0.00, 'a': 27.32, 'b': 0.0033}     # information for H2
data['CO']  = {'Hf': -110.70, 'a': 29.61, 'b': 0.0030}     # information for CO
data['CO2'] = {'Hf': -394.09, 'a': 44.32, 'b': 0.0073}     # information for CO2
data['O2']  =	{'Hf':    0.00, 'a': 30.51, 'b': 0.0035}     # information for O2


The enthalpy of a stream can be calculated as given below.  Note that this code is very similar to the one we developed in the [Jupyter notebook](https://colab.research.google.com/github/mjksill/CP213-online/blob/master/notebooks/lecture_20260923.ipynb) for the last lecture.



In [ ]:
instream = {'H2':1, 'O2':0.5}   # moles of each component in the inlet stream
outstream = {'H2O':1}           # moles of each component in the outlet stream


T0 = 298.15  # reference temperature
T = 600      # stream temperature
stream = outstream   # select the stream

# Note that in this calculation, the stream enthalpy is reported in kJ.
Hstream = 0
for mol, N in stream.items():
  Hf = data[mol]['Hf']
  a = data[mol]['a']
  b = data[mol]['b']
  H = Hf + (a*(T-T0) + 0.5*b*(T**2-T0**2))/1000    # Be careful with units!!!
  Hstream = Hstream + N * H
  print(f'{mol}: {N} mol, {H:.2f} kJ/mol')

print(f'stream enthalpy: {Hstream:.2f} kJ')

Rather than having to cut and paste the code to calculate the enthalpy of different streams, we can place the code within a function, which we call `get_Hstream`.  Note that the variables that we which to change to perforrm different calculations are used as the arguments of the function.  

Now we can calculate the heat $Q$ released from a reactor with an inlet stream of hydrogen and oxygen at a temperature $T$ and an outlet stream of water vapor at temperature $T_{\rm out}$ rather easily.

In [ ]:
def get_Hstream(T, stream):

    Hstream = 0
    for mol, N in stream.items():
        Hf = data[mol]['Hf']
        a = data[mol]['a']
        b = data[mol]['b']
        H = Hf + (a*(T-T0) + 0.5*b*(T**2-T0**2))/1000    # Be careful with units!!!
        Hstream = Hstream + N * H
#        print(f'{mol}: {N} mol, {H:.2f} kJ/mol')
    return Hstream


# calculate the enthalpy of a mixture of CO2 and water
Hin = get_Hstream(300, {'CO2': 1, 'H2O':4})


instream = {'H2':1, 'O2':0.5}  # inlet stream composition
outstream = {'H2O':1}          # outlet stream composition
T = 300      # temperature of the inlet stream
Tout = 4000  # temperature of the outlet stream

Hin = get_Hstream(T, instream)       # enthalpy  of the inlet stream
Hout = get_Hstream(Tout, outstream)  # enthalpy of the outlet stream
Q = Hin - Hout   # heat released by the reactor
print(f'heat released by reactor: {Q:.2f} kJ')

## Adiabatic flame temperature

Functions allow us to build complex calculations from several simpler calculations.  For example, we can create a function called `f` that calculates the heat released by the reactor, given that its outlet temperature is `x`.  Note that stream compositions and the inlet stream temperature are taken from definitions outside the function.  

If we want to find the adiabatic flame tempeerature of the inlet stream, we need to determine the temperature of the outlet stream that corresponds to the case where not heat is released from the reactor.  We can do this by using an equation solver to determine the value of `x` that makes the function `f` zero.  For more details on equation solvers, see the Jupyter notebook [Numerically solving algebraic equations](https://colab.research.google.com/github/mjksill/CP213-online/blob/master/notebooks/python_solver.ipynb) in the Week 2 learning materials on Myplace.


In [ ]:
instream = {'H2':1, 'O2':0.5}  # inlet stream composition
outstream = {'H2O':1}          # outlet stream composition
T = 300      # temperature of the inlet stream

# The function f calculates the heat released by the reactor for an outlet
# stream at temperature x.  Note that the inlet stream temperature T and stream
# compositions are taken from outside the function.
def f(x):
  Hin = get_Hstream(T, instream)
  Hout = get_Hstream(x, outstream)
  Q = Hin - Hout
  return Q



from scipy.optimize import bisect

xmin = 500
xmax = 6000
solution = bisect(f, a=xmin, b=xmax)

print(f'adiabatic flame temperature: {solution:.2f} K')


Finally, we note that we can introduce functions in order to allow future flexibility in the calculations.  For example, we can define the function `get_H` to determine the molar enthalpy of a compound.

In [ ]:
def get_H(T, mol):
    Hf = data[mol]['Hf']
    a = data[mol]['a']
    b = data[mol]['b']
    H = Hf + (a*(T-T0) + 0.5*b*(T**2-T0**2))/1000    # Be careful with units!!!
    return H


def get_Hstream(T, stream):
    Hstream = 0
    for mol, N in stream.items():
        H = get_H(T, mol)
        Hstream = Hstream + N * H
    return Hstream

T = 300
Tout = 4000
Hin = get_Hstream(T, instream)
Hout = get_Hstream(Tout, outstream)
Q = Hin - Hout
print(Q)



## Replacing the thermodynamic model

In the previous section, we worked with a relatively simple expression for the heat capacity and enthalpy, assuming that the heat capacity was a linear function of temperature.  If we have a more complicated expression for the thermodynamic quantities, such as using the Shomate equation for the heat capacity, we can still use all the code we developed in our previous work and just use new functions for `get_Cp` and `get_H` which correspond to the new thermodynamics model.



### Data

In [ ]:
data = {}

params = [{'Tmin': 298.0, 'Tmax': 1300.0, 'A': -0.703029, 'B': 108.4773, 'C': -42.52157, 'D': 5.862788, 'E': 0.678565, 'F': -76.84376, 'G': 158.7163, 'H': -74.8731},
          {'Tmin': 1300.0, 'Tmax': 6000.0, 'A': 85.81217, 'B': 11.26467, 'C': -2.114146, 'D': 0.13819, 'E': -26.42221, 'F': -153.5327, 'G': 224.4143, 'H': -74.8731}]
data['CH4'] = {'Hf': -74.87, 'param_list': params}

params = [{'Tmin': 298.0, 'Tmax': 1200.0, 'A': 24.99735, 'B': 55.18696, 'C': -33.69137, 'D': 7.948387, 'E': -0.136638, 'F': -403.6075, 'G': 228.2431, 'H': -393.5224},
          {'Tmin': 1200.0, 'Tmax': 6000.0, 'A': 58.16639, 'B': 2.720074, 'C': -0.492289, 'D': 0.038844, 'E': -6.447293, 'F': -425.9186, 'G': 263.6125, 'H': -393.5224}]
data['CO2'] = {'Hf': -393.52, 'param_list': params}

params = [{'Tmin': 298.0, 'Tmax': 1300.0, 'A': 25.56759, 'B': 6.09613, 'C': 4.054656, 'D': -2.671301, 'E': 0.131021, 'F': -118.0089, 'G': 227.3665, 'H': -110.5271},
          {'Tmin': 1300.0, 'Tmax': 6000.0, 'A': 35.1507, 'B': 1.300095, 'C': -0.205921, 'D': 0.01355, 'E': -3.28278, 'F': -127.8375, 'G': 231.712, 'H': -110.5271}]
data['CO'] = {'Hf': -110.53, 'param_list': params}

params = [{'Tmin': 500.0, 'Tmax': 1700.0, 'A': 30.092, 'B': 6.832514, 'C': 6.793435, 'D': -2.53448, 'E': 0.082139, 'F': -250.881, 'G': 223.3967, 'H': -241.8264},
          {'Tmin': 1700.0, 'Tmax': 6000.0, 'A': 41.96426, 'B': 8.622053, 'C': -1.49978, 'D': 0.098119, 'E': -11.15764, 'F': -272.1797, 'G': 219.7809, 'H': -241.8264}]
data['H2O'] = {'Hf': -241.83, 'param_list': params}

params = [{'Tmin': 100.0, 'Tmax': 700.0, 'A': 31.32234, 'B': -20.23531, 'C': 57.86644, 'D': -36.50624, 'E': -0.007374, 'F': -8.903471, 'G': 246.7945, 'H': 0.0},
          {'Tmin': 700.0, 'Tmax': 2000.0, 'A': 30.03235, 'B': 8.772972, 'C': -3.988133, 'D': 0.788313, 'E': -0.741599, 'F': -11.32468, 'G': 236.1663, 'H': 0.0},
          {'Tmin': 2000.0, 'Tmax': 6000.0, 'A': 20.91111, 'B': 10.72071, 'C': -2.020498, 'D': 0.146449, 'E': 9.245722, 'F': 5.337651, 'G': 237.6185, 'H': 0.0}]
data['O2'] = {'Hf': -0.0, 'param_list': params}

params = [{'Tmin': 100.0, 'Tmax': 500.0, 'A': 28.98641, 'B': 1.853978, 'C': -9.647459, 'D': 16.63537, 'E': 0.000117, 'F': -8.671914, 'G': 226.4168, 'H': 0.0},
          {'Tmin': 500.0, 'Tmax': 2000.0, 'A': 19.50583, 'B': 19.88705, 'C': -8.598535, 'D': 1.369784, 'E': 0.527601, 'F': -4.935202, 'G': 212.39, 'H': 0.0},
          {'Tmin': 2000.0, 'Tmax': 6000.0, 'A': 35.51872, 'B': 1.128728, 'C': -0.196103, 'D': 0.014662, 'E': -4.55376, 'F': -18.97091, 'G': 224.981, 'H': 0.0}]
data['N2'] = {'Hf': -0.0, 'param_list': params}

params = [{'Tmin': 298.0, 'Tmax': 1000.0, 'A': 33.066178, 'B': -11.363417, 'C': 11.432816, 'D': -2.772874, 'E': -0.158558, 'F': -9.980797, 'G': 172.707974, 'H': 0.0},
          {'Tmin': 1000.0, 'Tmax': 2500.0, 'A': 18.563083, 'B': 12.257357, 'C': -2.859786, 'D': 0.268238, 'E': 1.97799, 'F': -1.147438, 'G': 156.288133, 'H': 0.0},
          {'Tmin': 2500.0, 'Tmax': 6000.0, 'A': 43.41356, 'B': -4.293079, 'C': 1.272428, 'D': -0.096876, 'E': -20.533862, 'F': -38.515158, 'G': 162.081354, 'H': 0.0}]
data['H2'] = {'Hf': -0.0, 'param_list': params}



### Thermodynamic functions

For the Shomate equation, the heat capacity is given by:
$$
C_p(T)
=
A + Bt + Ct^2 + Dt^3 + Et^{-2}
$$
where $t=T/1000$, $T$ is the absolute temperature in kelvin.  The enthalpy is given by
$$
H(T)
=
H_f
+ At + Bt^2/2 + Ct^3/3 + Dt^4/4 - E/t + F - H
$$
where $H_f$ is the enthalpy of formation.

In the code block below, we implement the heat capacity and the enthalpy corresponding to the Shomate equation:


In [ ]:
# heat capacity J mol^{-1} K^{-1}
def get_Cp(T, mol):
    params = data[mol]['param_list']
    for param in params:
        if (T >= param['Tmin'] and T <= param['Tmax']):
            A = param['A']
            B = param['B']
            C = param['C']
            D = param['D']
            E = param['E']
            F = param['F']
            G = param['G']
            H = param['H']
            t = T/1000
            return A + B*t + C*t**2 + D*t**3 + E/t**2
    return 'error'


# enthalpy kJ mol^{-1}
def get_H(T, mol):
    Hf = data[mol]['Hf']   # the heat of formation is in kJ mol^{-1}
    params = data[mol]['param_list']
    for param in params:
        if (T >= param['Tmin'] and T <= param['Tmax']):
            A = param['A']
            B = param['B']
            C = param['C']
            D = param['D']
            E = param['E']
            F = param['F']
            G = param['G']
            H = param['H']
            t = T/1000
            DeltaH = A*t + B*t**2/2 + C*t**3/3 + D*t**4/4 - E/t + F - H
            return Hf + DeltaH
    print('error: temperature is out of bounds T={T} K')
    exit(0)
    return 'error'


In [ ]:
import numpy as np
import matplotlib.pylab as plt


T = 600
print(get_Cp(T, 'H2O'))
print(get_H(T, 'H2O'))




We can use the same code from before, but now with a more complicated thermodynamic model.

In [ ]:

def get_Hstream(T, stream):
    Hstream = 0
    for mol, N in stream.items():
        H = get_H(T, mol)
        Hstream = Hstream + N * H
    return Hstream

T = 300


instream = {'H2':1, 'O2':0.5}
outstream = {'H2O':1}
Tout = 4000
Hin = get_Hstream(T, instream)
Hout = get_Hstream(Tout, outstream)
Q = Hin - Hout
print(Q)

In [ ]:
instream = {'H2':1, 'O2':0.5}  # inlet stream composition
outstream = {'H2O':1}          # outlet stream composition
T = 300      # temperature of the inlet stream

# The function f calculates the heat released by the reactor for an outlet
# stream at temperature x.  Note that the inlet stream temperature T and stream
# compositions are taken from outside the function.
def f(x):
  Hin = get_Hstream(T, instream)
  Hout = get_Hstream(x, outstream)
  Q = Hin - Hout
  return Q



from scipy.optimize import bisect

xmin = 500
xmax = 6000
solution = bisect(f, a=xmin, b=xmax)

print(f'adiabatic flame temperature: {solution:.2f} K')